In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd
from docling.chunking import HybridChunker
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_qdrant import QdrantVectorStore
from langgraph.prebuilt import create_react_agent

load_dotenv()

True

### Without RAG

In [2]:
MODEL = "qwen3.8-flash"  # cheap + good for learning
llm = ChatOpenAI(model=MODEL, 
                temperature=0,
                base_url="https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1",
                extra_body={
                    "enable_thinking": False,
                })

SYSTEM_PROMPT = """You are Tilik, a helpful statistics assistant for Indonesian BPS data.
Rules:
- If you need a number from our sample dataset, call the tool; do not invent values.
- Answer in clear English unless the user writes in Indonesian.
- Keep answers short (2-4 sentences) unless asked for detail.
"""

USER_QUESTION = "Provinsi di indonesia dengan tingkat pengangguran terbuka tertinggi?"

messages = [
    SystemMessage(content=SYSTEM_PROMPT),
    HumanMessage(content=USER_QUESTION),
]

llm.invoke(messages)

AIMessage(content='I do not have access to real-time or current statistical data from BPS Indonesia. Please consult the latest official publication from Badan Pusat Statistik (BPS) for the most accurate and up-to-date information on open unemployment rates by province.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 97, 'total_tokens': 144, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 97}}, 'model_name': 'qwen3.8-flash', 'system_fingerprint': None, 'id': 'chatcmpl-b2f0788e-b5e8-4c5c-9ff5-c3da65c690f4', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--01a0c261-fe0b-7482-a4bf-69c81396cac9-0', usage_metadata={'input_tokens': 97, 'output_tokens': 47, 'total_tokens': 144, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})

### With RAG

In [ ]:
pdf_path = "../data/docs/statistik-demografi-indonesia--hasil-sensus-penduduk-2020.pdf"

# Born-digital FPDF: use the text layer. Default OCR only hits the 94 decorative images per page.
pipeline_options = PdfPipelineOptions(do_ocr=False, do_table_structure=True)
converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options),
    }
)
result = converter.convert(pdf_path)
doc = result.document

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [4]:
print("status:", result.status)
print("errors:", result.errors)
print("pages processed:", len(result.pages))

print("texts:", len(result.document.texts))
print("tables:", len(result.document.tables))
print("pictures:", len(result.document.pictures))

md = result.document.export_to_markdown()
print("markdown chars:", len(md))
print(repr(md[:500]))

status: ConversionStatus.SUCCESS
errors: []
pages processed: 191
texts: 2550
tables: 23
pictures: 239
markdown chars: 470047
'<!-- image -->\n\n(Hasil Sensus Penduduk 2020)\n\nhttps://www.bps.go.id\n\n<!-- image -->\n\nhttps://www.bps.go.id\n\n## (Hasil Sensus Penduduk 2020) https://www.bps.go.id\n\n<!-- image -->\n\n## Statistik Demografi Indonesia\n\n(Hasil Sensus Penduduk 2020)\n\nKatalog\n\n: 2201007\n\nNomor Publikasi : 04100.25001\n\nUkuran Buku\n\n: 17,6 cm X 25 cm\n\nJumlah Halaman\n\n: xviii+16 8 halaman\n\n## Penyusun Naskah :\n\nDirektorat Statistik Kependudukan dan Ketenagakerjaan\n\n## Penyunting :\n\nDirektorat Statistik Kependudukan dan Kete'


In [5]:
from collections import Counter

print(Counter(t.label for t in doc.texts))
print(Counter(t.label for t in doc.tables))

t = doc.texts[50]
print(t.label)
print(t.text[:400])
print(t.prov)                    # list of ProvenanceItem
print(t.prov[0].page_no, t.prov[0].bbox)
print(t.model_dump(mode="json")) # full JSON for one item

Counter({<DocItemLabel.TEXT: 'text'>: 2049, <DocItemLabel.PAGE_FOOTER: 'page_footer'>: 301, <DocItemLabel.SECTION_HEADER: 'section_header'>: 85, <DocItemLabel.LIST_ITEM: 'list_item'>: 52, <DocItemLabel.CAPTION: 'caption'>: 50, <DocItemLabel.FORMULA: 'formula'>: 13})
Counter({<DocItemLabel.TABLE: 'table'>: 16, <DocItemLabel.DOCUMENT_INDEX: 'document_index'>: 7})
section_header
halaman
[ProvenanceItem(page_no=12, bbox=BoundingBox(l=398.4909, t=602.4272, r=435.5909, b=593.8372435255712, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 7))]
12 l=398.4909 t=602.4272 r=435.5909 b=593.8372435255712 coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>
{'self_ref': '#/texts/50', 'parent': {'cref': '#/body'}, 'children': [], 'content_layer': 'body', 'meta': None, 'label': 'section_header', 'prov': [{'page_no': 12, 'bbox': {'l': 398.4909, 't': 602.4272, 'r': 435.5909, 'b': 593.8372435255712, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 7]}], 'orig': 'halaman', 'text': 'halaman'

In [13]:
t = doc.tables[10]

print(t.label)
print("page:", t.prov[0].page_no if t.prov else None)
print("captions refs:", t.captions)
print("caption text:", t.caption_text(doc))   # often ""
print("footnotes:", t.footnotes)

# title is usually the previous text item on the same page
page = t.prov[0].page_no
nearby = [
    x for x in doc.texts
    if x.prov and x.prov[0].page_no == page
]
for x in nearby:
    print(x.label, x.text[:120])

table
page: 176
captions refs: [RefItem(cref='#/texts/2508')]
caption text: Lampiran 4		 Laju Pertumbuhan Penduduk Indonesia Menurut Provinsi, 1971 - 1980, 1980 - 1990, 1990 - 2000, 2000 - 2020, dan 2010 - 2020
footnotes: []
caption Lampiran 4		 Laju Pertumbuhan Penduduk Indonesia Menurut Provinsi, 1971 - 1980, 1980 - 1990, 1990 - 2000, 2000 - 2020, d
page_footer 156
page_footer Statistik Demografi Indonesia (Hasil Long Form Sensus Penduduk 2020)


In [12]:
print(doc.tables[10].export_to_markdown())

Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.


| Provinsi                  | 1971-1980   | 1980-1990                  | 1990-2000   | 2000-2010   | 2010-2020   |
|---------------------------|-------------|----------------------------|-------------|-------------|-------------|
| (1)                       | (2)         | (3)                        | (4)         | (5)         | (6)         |
| Aceh                      | 2,93        | 2,72                       | 1,46        | 2,36 1      | 1,56        |
| Sumatera Utara            | 2,60        | 2,06                       | 1,32        | 1,10        | 1,28        |
| Sumatera Barat            | 2,21        | 1,62                       | 0,63        | 1,34        | 1,29        |
| Riau                      | 3,11        | 4,30                       | 4,35        | 3,58        | 1,40        |
| Jambi                     | 4,07        | 3,40                       | 1,84        | 2,56        | 1,34        |
| Sumatera Selatan          | 3,32        | 3,15                       | 2,39   

In [16]:
chunker = HybridChunker()
chunks = list(chunker.chunk(dl_doc=doc))

docs = []
for chunk in chunks:
    text = chunker.contextualize(chunk)
    if not text.strip():
        continue
    docs.append(
        Document(
            page_content=text,
            metadata={
                "source": pdf_path,
                "headings": getattr(chunk.meta, "headings", None),
            },
        )
    )
print(len(docs), "chunks")

Token indices sequence length is longer than the specified maximum sequence length for this model (533 > 512). Running this sequence through the model will result in indexing errors


889 chunks


In [18]:
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
# Same object for QdrantVectorStore — no wrapper
dim = len(embeddings.embed_query("Apa itu TPK?"))
print(EMBEDDING_MODEL, "dim =", dim)  # 384

sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 dim = 384


In [19]:
COLLECTION = "bps-sp2020-demografi"

vector_store = QdrantVectorStore.from_documents(
    documents=docs,
    embedding=embeddings,
    url="http://localhost:6333",
    collection_name=COLLECTION,
)

In [20]:
vector_store.similarity_search("laju pertumbuhan penduduk menurut provinsi", k=3)

[Document(metadata={'source': '../data/docs/statistik-demografi-indonesia--hasil-sensus-penduduk-2020.pdf', 'headings': ['Keterangan:'], '_id': '9655d85b-7dee-4f86-80a7-267341fb075a', '_collection_name': 'bps-sp2020-demografi'}, page_content='Keterangan:\nLaju pertumbuhan penduduk yang cepat merupakan salah satu gejala dari pembangunan yang  lambat.  Berdasarkan  gambar  2.6,  provinsi  dengan  laju  pertumbuhan  penduduk tercepat pada 2010-2020 antara lain Provinsi Papua (4,13 persen), Provinsi Papua Barat (3,94  persen),  dan  Provinsi  Kalimantan  Utara  (2,86  persen).  Provinsi-provinsi  tersebut memerlukan anggaran yang besar dalam memberantas kemiskinan, mengakhiri kelaparan dan  kekurangan  gizi,  serta  memastikan  kemudahan  akses  menuju  layanan  kesehatan, pendidikan, dan layanan lainnya. Untuk mencapai tujuan-tujuan tersebut, pertumbuhan penduduk  yang  cepat  menambah  urgensi  untuk  memperbesar  skala  investasi'),
 Document(metadata={'source': '../data/docs/statistik-

In [25]:
retriever = vector_store.as_retriever(search_kwargs={"k": 5})

RAG_SYSTEM = """You are Tilik, a helpful statistics assistant for Indonesian BPS data.
Use only the retrieved excerpts from Statistik Demografi Indonesia (SP2020).
If the excerpts do not contain the answer, say so. Do not invent numbers.
Quote the figure and the table or section name when possible.
Answer in the same language as the question. Keep answers short unless asked for detail.
"""

# This PDF is census demography, not Sakernas — ask something in the book.
USER_QUESTION = "Provinsi mana yang laju pertumbuhan penduduknya paling cepat pada 2010-2020?"

hits = retriever.invoke(USER_QUESTION)
context = "\n\n---\n\n".join(hit.page_content for hit in hits)

messages = [
    SystemMessage(content=RAG_SYSTEM),
    HumanMessage(content=f"Context:\n{context}\n\nQuestion: {USER_QUESTION}"),
]
llm.invoke(messages)

AIMessage(content='Berdasarkan Gambar 2.6, provinsi dengan laju pertumbuhan penduduk tercepat pada periode 2010-2020 adalah **Provinsi Papua** dengan angka **4,13 persen**.\n\nDisusul oleh Provinsi Papua Barat (3,94 persen) dan Provinsi Kalimantan Utara (2,86 persen).', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 69, 'prompt_tokens': 854, 'total_tokens': 923, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 854}}, 'model_name': 'qwen3.8-flash', 'system_fingerprint': None, 'id': 'chatcmpl-cd4865e8-9c7a-487f-91f1-a24726626846', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--01a0c389-21ef-7311-b164-29df67fb6b59-0', usage_metadata={'input_tokens': 854, 'output_tokens': 69, 'total_tokens': 923, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})

In [24]:
USER_QUESTION = "Apa aja yang bisa kutanyakan?"
messages = [
    SystemMessage(content=RAG_SYSTEM),
    HumanMessage(content=f"Context:\n{context}\n\nQuestion: {USER_QUESTION}"),
]
print(llm.invoke(messages).content)

Berdasarkan data Statistik Demografi Indonesia (SP2020) yang tersedia, Anda dapat menanyakan hal-hal berikut:

1.  **Laju Pertumbuhan Penduduk:**
    *   Provinsi mana saja yang memiliki laju pertumbuhan penduduk tercepat pada periode 2010-2020?
    *   Berapa persentase laju pertumbuhan penduduk untuk Provinsi Papua, Papua Barat, dan Kalimantan Utara?
    *   Bagaimana tren laju pertumbuhan penduduk Indonesia dari tahun 1961 hingga 2020?

2.  **Dampak Pertumbuhan Penduduk terhadap Pembangunan:**
    *   Apa tantangan yang dihadapi daerah dengan sumber daya terbatas akibat laju pertumbuhan penduduk yang cepat?
    *   Mengapa pertumbuhan penduduk yang cepat menambah urgensi memperbesar skala investasi?
    *   Bagaimana peran kualitas penduduk (IPM) dalam meningkatkan produktivitas dan pertumbuhan ekonomi?

3.  **Proyeksi Penduduk:**
    *   Bagaimana gambaran proyeksi jumlah penduduk Provinsi Papua Barat Daya untuk tahun 2021-2023 berdasarkan SP2020?
    *   Apakah distribusi penduduk

### Adaptive RAG